In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Đọc dữ liệu Iris từ file OpenML
data, meta = arff.loadarff("dataset_61_iris.arff")
df = pd.DataFrame(data)

# Chuyển dữ liệu chữ về dạng bình thường
for col in df.select_dtypes(["object"]).columns:
    df[col] = df[col].str.decode("utf-8")

print("THÔNG TIN BỘ DỮ LIỆU IRIS")
print("Kích thước dữ liệu:", df.shape)
print("\n5 dòng đầu:")
print(df.head())

print("\nThông tin dữ liệu:")
print(df.info())

print("\nSố giá trị bị thiếu:")
print(df.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_61_iris.arff'

In [ ]:
# Kiểm tra và xóa dữ liệu trùng
print("Số dòng trùng:", df.duplicated().sum())
df = df.drop_duplicates()

# Tách đặc trưng và nhãn
X = df.drop("class", axis=1)
y = df["class"]

# Xem số lượng mẫu của từng lớp
print("\nPhân bố các lớp:")
print(y.value_counts())

# Vẽ biểu đồ phân bố lớp
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="class")
plt.title("Phân bố các lớp hoa Iris")
plt.show()

# Xem mối quan hệ giữa các đặc trưng
sns.pairplot(df, hue="class")
plt.show()

# Xem tương quan giữa các đặc trưng
plt.figure(figsize=(7, 5))
sns.heatmap(X.corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Ma trận tương quan")
plt.show()

# Chia tập train và test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Số mẫu train:", len(X_train))
print("Số mẫu test:", len(X_test))

In [ ]:
# Các mô hình phân loại
models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ),

    "KNN": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),

    "SVM": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        SVC(kernel="rbf")
    ),

    "Decision Tree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(random_state=42)
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=100, random_state=42)
    ),

    "Naive Bayes": make_pipeline(
        SimpleImputer(strategy="median"),
        GaussianNB()
    ),

    "Gradient Boosting": make_pipeline(
        SimpleImputer(strategy="median"),
        GradientBoostingClassifier(random_state=42)
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    # Dự đoán trên tập train và test
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Overfitting Gap": train_acc - test_acc
    })

# So sánh kết quả các mô hình
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Test Accuracy", ascending=False
)

print("KẾT QUẢ SO SÁNH CÁC MÔ HÌNH")
print(results_df.round(4).to_string(index=False))

# Vẽ biểu đồ so sánh
results_df.set_index("Model")[
    ["Train Accuracy", "Test Accuracy"]
].plot(kind="bar", figsize=(11, 5))

plt.title("So sánh độ chính xác của các mô hình")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

In [3]:
# So sánh Decision Tree trước và sau khi giới hạn độ sâu
tree_models = {
    "Decision Tree gốc": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(random_state=42)
    ),

    "Decision Tree giới hạn độ sâu": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=5,
            random_state=42
        )
    )
}

print("KIỂM TRA OVERFITTING")

for name, model in tree_models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print("\n", name)
    print(f"Train Accuracy: {train_acc * 100:.2f}%")
    print(f"Test Accuracy : {test_acc * 100:.2f}%")
    print(f"Khoảng cách   : {(train_acc - test_acc) * 100:.2f}%")

KIỂM TRA OVERFITTING


NameError: name 'X_train' is not defined